# Workshop 2 — YOLOv8 Fine-tuning & Deployment
**Materia:** Inteligencia Artificial  
**Dataset:** [Logistics (Roboflow Universe)](https://universe.roboflow.com/large-benchmark-datasets/logistics-sz9jr/browse)

## Part 0 — Setup

In [ ]:
!pip install ultralytics==8.* litserve==0.* fastapi uvicorn pillow opencv-python supervision roboflow -q

## Part 1 — Dataset Download

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_KEY")  # reemplaza con tu API key
project = rf.workspace("large-benchmark-datasets").project("logistics-sz9jr")
dataset = project.version(2).download("yolov8", location="datasets/logistics")
print("Dataset path:", dataset.location)

## Part 2 — Fine-tune YOLOv8s

**Decisión de modelo:** Se usa `yolov8s` en lugar de `yolov8n` porque el dataset de logística contiene objetos pequeños y variados (cajas, pallets, etiquetas). El modelo `s` (small) tiene ~11M parámetros vs ~3M del `n`, lo que le da mayor capacidad de representación sin ser prohibitivo en tiempo de entrenamiento.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # yolov8s: mejor balance capacidad/velocidad para objetos pequeños

results = model.train(
    data="datasets/logistics/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,          # lr final = lr0 * lrf
    warmup_epochs=3,
    hsv_h=0.015,        # augmentación: hue
    hsv_s=0.7,          # augmentación: saturación
    hsv_v=0.4,          # augmentación: valor/brillo
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/logistics",
    name="y8s",
    exist_ok=True,
)

print("Best weights:", results.save_dir)

## Part 3 — Validación y Métricas

In [ ]:
from ultralytics import YOLO
import pandas as pd

best_model = YOLO("runs/logistics/y8s/weights/best.pt")

# --- Validación en val set ---
val_metrics = best_model.val(data="datasets/logistics/data.yaml", split="val")

print("=== VAL SET ===")
print(f"mAP@0.50:       {val_metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95:  {val_metrics.box.map:.4f}")
print(f"Precision:      {val_metrics.box.mp:.4f}")
print(f"Recall:         {val_metrics.box.mr:.4f}")

# F1 = 2 * P * R / (P + R)
P = val_metrics.box.mp
R = val_metrics.box.mr
F1 = 2 * P * R / (P + R + 1e-9)
print(f"F1 Score:       {F1:.4f}")

In [ ]:
# --- Validación en test set ---
test_metrics = best_model.val(data="datasets/logistics/data.yaml", split="test")

print("=== TEST SET ===")
print(f"mAP@0.50:       {test_metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95:  {test_metrics.box.map:.4f}")
print(f"Precision:      {test_metrics.box.mp:.4f}")
print(f"Recall:         {test_metrics.box.mr:.4f}")

P_t = test_metrics.box.mp
R_t = test_metrics.box.mr
F1_t = 2 * P_t * R_t / (P_t + R_t + 1e-9)
print(f"F1 Score:       {F1_t:.4f}")

In [ ]:
# --- AP y AR por clase ---
names = best_model.names
ap_per_class = val_metrics.box.ap_class_index  # índices de clases
ap50_per_class = val_metrics.box.ap50          # AP@0.50 por clase
ap_per_class_vals = val_metrics.box.ap         # AP@0.50:0.95 por clase

print("\n=== AP por clase (val) ===")
for idx, ap50, ap in zip(ap_per_class, ap50_per_class, ap_per_class_vals):
    print(f"  {names[idx]:20s}  AP@50={ap50:.4f}  AP@50:95={ap:.4f}")

In [ ]:
# --- Curvas de entrenamiento ---
import matplotlib.pyplot as plt
import pandas as pd

results_csv = pd.read_csv("runs/logistics/y8s/results.csv")
results_csv.columns = results_csv.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(results_csv["epoch"], results_csv["train/box_loss"], label="train")
axes[0].plot(results_csv["epoch"], results_csv["val/box_loss"], label="val")
axes[0].set_title("Box Loss")
axes[0].legend()

axes[1].plot(results_csv["epoch"], results_csv["metrics/mAP50(B)"])
axes[1].set_title("mAP@0.50")

axes[2].plot(results_csv["epoch"], results_csv["metrics/mAP50-95(B)"])
axes[2].set_title("mAP@0.50:0.95")

plt.tight_layout()
plt.savefig("runs/logistics/y8s/training_curves.png", dpi=150)
plt.show()

## Part 4 — Export

In [ ]:
best_model.export(format="onnx")
print("Modelo exportado a ONNX")